In [1]:
import sys
import logging
from typing import Dict, Any, List, Optional, Set
from dataclasses import dataclass, field
from enum import Enum
import uuid

# 配置日志输出
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("SideEffectRuntime")

# ============================================================================
# 1. 动作契约定义 (ActionContract)
# ============================================================================

class ActionType(str, Enum):
    READ = "READ"
    WRITE_SIDE_EFFECT = "WRITE_SIDE_EFFECT"  # 具有副作用的写操作（如转账、审批）

@dataclass
class ActionContract:
    tool_name: str
    action_type: ActionType
    required_role: str = "EMPLOYEE"
    sensitive_parameters: Set[str] = field(default_factory=set) # 不可擅自修改的参数

# ============================================================================
# 2. 策略网关决策结果与 PolicyGate
# ============================================================================

class PolicyDecision(str, Enum):
    ALLOW = "ALLOW"
    DENY = "DENY"
    REQUIRE_CONFIRMATION = "REQUIRE_CONFIRMATION"

@dataclass
class PolicyResult:
    decision: PolicyDecision
    reason: str

class PolicyGate:
    @staticmethod
    def evaluate(
        user_role: str,
        contract: ActionContract,
        user_intent_goal: Dict[str, Any],
        agent_proposed_action: Dict[str, Any]
    ) -> PolicyResult:
        """
        综合评估：权限校验 + 副作用确认 + 意图防篡改 (Goal Mutation Guard)
        """
        # 1. 权限判定 (RBAC Check)
        role_hierarchy = {"EMPLOYEE": 1, "MANAGER": 2, "ADMIN": 3}
        user_level = role_hierarchy.get(user_role, 0)
        required_level = role_hierarchy.get(contract.required_role, 1)
        
        if user_level < required_level:
            return PolicyResult(
                decision=PolicyDecision.DENY,
                reason=f"权限不足: 用户角色 [{user_role}] 无法执行需要 [{contract.required_role}] 权限的操作。"
            )

        # 2. 纯只读操作且无越权 -> 直接放行
        if contract.action_type == ActionType.READ:
            return PolicyResult(decision=PolicyDecision.ALLOW, reason="只读操作，权限与意图匹配。")

        # 3. 针对具有副作用的操作 (Write Side Effect)
        # 3.1 检查动作是否匹配用户初始意图 (防止越权乱调用工具)
        expected_action = user_intent_goal.get("action")
        proposed_tool = agent_proposed_action.get("tool_name")
        if expected_action == "query_balance" and proposed_tool == "execute_transfer":
            return PolicyResult(
                decision=PolicyDecision.REQUIRE_CONFIRMATION,
                reason="意图越权拦截: 用户仅要求查询余额，Agent 擅自发起具有副作用的转账写操作。"
            )

        # 3.2 检查敏感参数是否发生静默篡改 (Goal Mutation Guard)
        for param in contract.sensitive_parameters:
            original_val = user_intent_goal.get(param)
            proposed_val = agent_proposed_action.get(param)
            if original_val is not None and proposed_val is not None:
                if original_val != proposed_val:
                    return PolicyResult(
                        decision=PolicyDecision.REQUIRE_CONFIRMATION,
                        reason=f"参数篡改拦截: 用户授权参数 [{param}: {original_val}] 被 Agent 修改为 [{param}: {proposed_val}]。"
                    )

        return PolicyResult(
            decision=PolicyDecision.ALLOW,
            reason="副作用写操作安全校验通过：权限符合，意图与参数一致。"
        )

# ============================================================================
# 3. 带幂等性保护的后端与 ToolRuntime (Idempotency & Timeout Safety)
# ============================================================================

@dataclass
class ExecutionResult:
    success: bool
    status_code: int
    data: Dict[str, Any]
    message: str

class MockBankingBackend:
    """模拟真实的银行/支付后端，内部维护交易账本和幂等记录"""
    def __init__(self):
        # 存储已执行成功的幂等记录: idempotency_key -> execution_result
        self.ledger: Dict[str, ExecutionResult] = {}
        self.account_balance = 5000.0

    def transfer(self, amount: float, idempotency_key: str, simulate_timeout: bool = False) -> ExecutionResult:
        # 【关键幂等防线】如果该 Key 已经处理过，直接返回历史结果，绝不重复扣款！
        if idempotency_key in self.ledger:
            logger.info(f"🛡️ [Idempotency Check] 捕获到重复提交的 Key: {idempotency_key}。直接返回历史账本结果，未重复扣款。")
            return self.ledger[idempotency_key]

        # 模拟网络 Timeout 异常
        if simulate_timeout:
            logger.error(f"💥 [Backend Fault] 支付网关发生 Socket Timeout！(但后端实际可能已经扣款)")
            # 即使发生 Timeout，为了防御，后端在底层也应当记录该 Key 的尝试状态或进行回滚
            # 本场景中模拟“下游实际已成功执行，但由于网络中断响应未送达”
            success_result = ExecutionResult(
                success=True,
                status_code=200,
                data={"balance": self.account_balance - amount, "transferred": amount},
                message="SUCCESS_AFTER_TIMEOUT_RECOVERY"
            )
            # 记录到账本
            self.ledger[idempotency_key] = success_result
            raise TimeoutError("Gateway Read Timeout - Response Dropped in Transit")

        # 正常扣款逻辑
        if amount > self.account_balance:
            res = ExecutionResult(success=False, status_code=400, data={}, message="DECLINED: INSUFFICIENT_FUNDS")
            return res

        self.account_balance -= amount
        res = ExecutionResult(
            success=True,
            status_code=200,
            data={"balance": self.account_balance, "transferred": amount},
            message="SUCCESS"
        )
        # 写入幂等账本
        self.ledger[idempotency_key] = res
        return res

    def query_status(self, idempotency_key: str) -> Optional[ExecutionResult]:
        """通过幂等键查询下游状态"""
        return self.ledger.get(idempotency_key)


class ToolRuntime:
    def __init__(self, backend: MockBankingBackend):
        self.backend = backend

    def execute_with_side_effect_guard(
        self,
        action_name: str,
        params: Dict[str, Any],
        idempotency_key: str,
        simulate_timeout: bool = False
    ) -> ExecutionResult:
        """
        安全的副作用执行器：
        当写操作遭遇 Timeout 时，决不盲目重新调用发送交易，而是通过查询幂等状态恢复。
        """
        logger.info(f"🚀 [ToolRuntime] 执行写操作: {action_name} | IdempotencyKey: {idempotency_key}")
        
        try:
            # 首次尝试调用
            return self.backend.transfer(
                amount=params.get("amount", 0.0),
                idempotency_key=idempotency_key,
                simulate_timeout=simulate_timeout
            )
        except TimeoutError as te:
            logger.warning(f"⚠️ [Runtime Catch] 捕获到网络超时错误: {te}")
            logger.info("🔄 [Safe Recovery Protocol] 触发容灾补偿：向后端查询该 IdempotencyKey 的实际执行状态，绝不盲目重试扣款！")
            
            # 安全恢复：调用幂等查询接口
            recovered_res = self.backend.query_status(idempotency_key)
            if recovered_res:
                logger.info(f"✅ [Recovery Success] 下游账本确认：该笔交易已安全落地，状态码: {recovered_res.status_code}")
                return recovered_res
            else:
                # 若账本无记录说明确实未到达后端，可安全重试（本案例中已在发生 timeout 前写入，故可找回）
                return ExecutionResult(success=False, status_code=504, data={}, message="TIMEOUT_AND_NO_LEDGER_RECORD")


# ============================================================================
# 4. 验证五个核心测试场景 (Verification Scenarios)
# ============================================================================

def run_day36_tests():
    print("=" * 70)
    print("🧪 Day 36: Side Effect Policy Gate & ToolRuntime Verification")
    print("=" * 70)

    backend = MockBankingBackend()
    runtime = ToolRuntime(backend)

    transfer_contract = ActionContract(
        tool_name="execute_transfer",
        action_type=ActionType.WRITE_SIDE_EFFECT,
        required_role="MANAGER",
        sensitive_parameters={"amount", "recipient"}
    )

    # ------------------------------------------------------------------------
    # 场景 1：用户只要求查询余额，Agent 擅自发起转账
    # ------------------------------------------------------------------------
    print("\n--- [Scenario 1] 用户只要求查询余额，Agent 擅自转账 ---")
    user_goal_1 = {"action": "query_balance"}
    agent_action_1 = {"tool_name": "execute_transfer", "amount": 1000}
    policy_res_1 = PolicyGate.evaluate("MANAGER", transfer_contract, user_goal_1, agent_action_1)
    print(f"Policy Decision : {policy_res_1.decision.value}")
    print(f"Reason          : {policy_res_1.reason}")

    # ------------------------------------------------------------------------
    # 场景 2：用户要求审批，但权限系统判定无权
    # ------------------------------------------------------------------------
    print("\n--- [Scenario 2] 用户要求审批，但权限系统判定无权 ---")
    user_goal_2 = {"action": "execute_transfer", "amount": 500}
    agent_action_2 = {"tool_name": "execute_transfer", "amount": 500}
    policy_res_2 = PolicyGate.evaluate("EMPLOYEE", transfer_contract, user_goal_2, agent_action_2) # 角色过低
    print(f"Policy Decision : {policy_res_2.decision.value}")
    print(f"Reason          : {policy_res_2.reason}")

    # ------------------------------------------------------------------------
    # 场景 3：用户授权转账 1000，Agent 擅自改成 2000
    # ------------------------------------------------------------------------
    print("\n--- [Scenario 3] 用户授权转账 1000，Agent 擅自改成 2000 ---")
    user_goal_3 = {"action": "execute_transfer", "amount": 1000, "recipient": "Alice"}
    agent_action_3 = {"tool_name": "execute_transfer", "amount": 2000, "recipient": "Alice"} # 金额被篡改
    policy_res_3 = PolicyGate.evaluate("MANAGER", transfer_contract, user_goal_3, agent_action_3)
    print(f"Policy Decision : {policy_res_3.decision.value}")
    print(f"Reason          : {policy_res_3.reason}")

    # ------------------------------------------------------------------------
    # 场景 4：用户授权转账 1000，权限通过，参数一致
    # ------------------------------------------------------------------------
    print("\n--- [Scenario 4] 用户授权转账 1000，权限通过，参数一致 ---")
    user_goal_4 = {"action": "execute_transfer", "amount": 1000, "recipient": "Alice"}
    agent_action_4 = {"tool_name": "execute_transfer", "amount": 1000, "recipient": "Alice"}
    policy_res_4 = PolicyGate.evaluate("MANAGER", transfer_contract, user_goal_4, agent_action_4)
    print(f"Policy Decision : {policy_res_4.decision.value}")
    print(f"Reason          : {policy_res_4.reason}")

    # ------------------------------------------------------------------------
    # 场景 5：转账请求 Timeout，但下游已执行成功（验证幂等与账本单次扣款）
    # ------------------------------------------------------------------------
    print("\n--- [Scenario 5] 转账 Timeout 容灾与幂等防重复扣款验证 ---")
    idempotency_key = str(uuid.uuid4())
    print(f"生成的唯一幂等键 (Idempotency Key): {idempotency_key}")
    print(f"初始后端账户余额: {backend.account_balance}")

    # 第一次调用：人为模拟触发 Timeout，但后端已经成功落地扣款
    print("\n>>> 第一次调用 (触发 Timeout 异常)...")
    res_first = runtime.execute_with_side_effect_guard(
        action_name="execute_transfer",
        params={"amount": 1000},
        idempotency_key=idempotency_key,
        simulate_timeout=True
    )
    print(f"运行时最终恢复结果: {res_first.message}, 下游返回余额: {res_first.data.get('balance')}")
    print(f"当前后端账户余额: {backend.account_balance}")

    # 第二次调用（模拟由于超时未收到响应而再次发起相同请求）
    print("\n>>> 第二次调用 (使用相同的 Idempotency Key 再次提交)...")
    res_second = runtime.execute_with_side_effect_guard(
        action_name="execute_transfer",
        params={"amount": 1000},
        idempotency_key=idempotency_key,
        simulate_timeout=False
    )
    print(f"第二次提交返回结果: {res_second.message}, 下游返回余额: {res_second.data.get('balance')}")
    print(f"最终后端账户余额 (应保持不变，证明仅扣款 1 次): {backend.account_balance}")
    print(f"后端 Ledger 账本中记录的交易总笔数: {len(backend.ledger)}")

if __name__ == "__main__":
    run_day36_tests()

2026-10-20 10:59:17,095 - [INFO] - 🚀 [ToolRuntime] 执行写操作: execute_transfer | IdempotencyKey: 329797c8-def5-4056-a91e-c6553a545d85
2026-10-20 10:59:17,096 - [ERROR] - 💥 [Backend Fault] 支付网关发生 Socket Timeout！(但后端实际可能已经扣款)
2026-10-20 10:59:17,097 - [WARNING] - ⚠️ [Runtime Catch] 捕获到网络超时错误: Gateway Read Timeout - Response Dropped in Transit
2026-10-20 10:59:17,098 - [INFO] - 🔄 [Safe Recovery Protocol] 触发容灾补偿：向后端查询该 IdempotencyKey 的实际执行状态，绝不盲目重试扣款！
2026-10-20 10:59:17,099 - [INFO] - ✅ [Recovery Success] 下游账本确认：该笔交易已安全落地，状态码: 200
2026-10-20 10:59:17,101 - [INFO] - 🚀 [ToolRuntime] 执行写操作: execute_transfer | IdempotencyKey: 329797c8-def5-4056-a91e-c6553a545d85
2026-10-20 10:59:17,104 - [INFO] - 🛡️ [Idempotency Check] 捕获到重复提交的 Key: 329797c8-def5-4056-a91e-c6553a545d85。直接返回历史账本结果，未重复扣款。


🧪 Day 36: Side Effect Policy Gate & ToolRuntime Verification

--- [Scenario 1] 用户只要求查询余额，Agent 擅自转账 ---
Policy Decision : REQUIRE_CONFIRMATION
Reason          : 意图越权拦截: 用户仅要求查询余额，Agent 擅自发起具有副作用的转账写操作。

--- [Scenario 2] 用户要求审批，但权限系统判定无权 ---
Policy Decision : DENY
Reason          : 权限不足: 用户角色 [EMPLOYEE] 无法执行需要 [MANAGER] 权限的操作。

--- [Scenario 3] 用户授权转账 1000，Agent 擅自改成 2000 ---
Policy Decision : REQUIRE_CONFIRMATION
Reason          : 参数篡改拦截: 用户授权参数 [amount: 1000] 被 Agent 修改为 [amount: 2000]。

--- [Scenario 4] 用户授权转账 1000，权限通过，参数一致 ---
Policy Decision : ALLOW
Reason          : 副作用写操作安全校验通过：权限符合，意图与参数一致。

--- [Scenario 5] 转账 Timeout 容灾与幂等防重复扣款验证 ---
生成的唯一幂等键 (Idempotency Key): 329797c8-def5-4056-a91e-c6553a545d85
初始后端账户余额: 5000.0

>>> 第一次调用 (触发 Timeout 异常)...
运行时最终恢复结果: SUCCESS_AFTER_TIMEOUT_RECOVERY, 下游返回余额: 4000.0
当前后端账户余额: 5000.0

>>> 第二次调用 (使用相同的 Idempotency Key 再次提交)...
第二次提交返回结果: SUCCESS_AFTER_TIMEOUT_RECOVERY, 下游返回余额: 4000.0
最终后端账户余额 (应保持不变，证明仅扣款 1 次): 5000.0
后端 Ledger 账本中记录的交易总笔数: 1
